In [ ]:
##### importing custom modules from the projects folder
import sys, os
from pathlib import Path
# Add project root to sys.path - search backwards through folders to find config.py
cwd = Path.cwd()
# Search upwards until a "config*" file is found
for parent in [cwd, *cwd.parents]:
    match = next(parent.glob('config*'), None)
    if match:
        PROJECT_ROOT = match.parent
        break
sys.path.append(str(PROJECT_ROOT))
import config
##### -------------------------------------------------


import json, time, requests, random
import pandas as pd
import numpy as np
from datetime import datetime
from bs4 import BeautifulSoup as bs
from sqlalchemy import create_engine

today = datetime.today().date()
season = '2025'

datetime.date(2025, 8, 24)

In [ ]:

url = 'https://api.bettingpros.com/v3/offers'

# prop_name:[market id, number of entries at the time running. have to manually lookup on site]
# i can't figure out how to get a response > 5 players at a time so the total number is required
# to loop through pages
market_ids = {
    'total_rushing_yds':[301, 57],
    'total_rushing_tds':[305, 57],
    'total_receiving_yds':[302, 105],
    'total_receiving_tds':[306, 87],
    'total_passing_yds':[300, 33],
    'total_passing_tds':[304, 32]
}

scraped_json = {
    'total_rushing_yds':[],
    'total_rushing_tds':[],
    'total_receiving_yds':[],
    'total_receiving_tds':[],
    'total_passing_yds':[],
    'total_passing_tds':[]
}
book_id = None

params = {
    'sport': 'NFL',
    'market_id':None,    # 'marketId'
    'season': season,    # 'YYYY'
    'book_id': 'null',   
    'limit': '5',
    'page': '1'
}

headers = {
    'Host': 'api.bettingpros.com',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Referer': 'https://www.bettingpros.com/',
    'Origin': 'https://www.bettingpros.com',
    'DNT': '1',
    'Sec-GPC': '1',
    'Connection': 'keep-alive',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site',
    'Priority': 'u=4',
    'TE': 'trailers',
    'x-api-key': 'CHi8Hy5CEE4khd46XNYL23dCFX96oUdw6qOt1Dnh'  # you’ll need to add this manually
}

for k,v in market_ids.items():
    
    # load market id for specific prop
    params['market_id'] = str(v[0])

    # have to loop through pages 5 players at a time
    n_players = v[1]
    n_pages = int(n_players / 5) + 1
    for i in range(1, n_pages + 1):

        params['page'] = i
        
        r = requests.get(
            url,
            headers=headers,
            params=params
        )
        print(k, i, r)
        try:
            #soup = bs(r.text, features='lxml')
            scraped_json[k].append(r.json())
            print('made soup')
        except:
            print('soup ruined')

        time.sleep(3)

total_rushing_yds 1 <Response [200]>
made soup
total_rushing_yds 2 <Response [200]>
made soup
total_rushing_yds 3 <Response [200]>
made soup
total_rushing_yds 4 <Response [200]>
made soup
total_rushing_yds 5 <Response [200]>
made soup
total_rushing_yds 6 <Response [200]>
made soup
total_rushing_yds 7 <Response [200]>
made soup
total_rushing_yds 8 <Response [200]>
made soup
total_rushing_yds 9 <Response [200]>
made soup
total_rushing_yds 10 <Response [200]>
made soup
total_rushing_yds 11 <Response [200]>
made soup
total_rushing_yds 12 <Response [200]>
made soup
total_rushing_tds 1 <Response [200]>
made soup
total_rushing_tds 2 <Response [200]>
made soup
total_rushing_tds 3 <Response [200]>
made soup
total_rushing_tds 4 <Response [200]>
made soup
total_rushing_tds 5 <Response [200]>
made soup
total_rushing_tds 6 <Response [200]>
made soup
total_rushing_tds 7 <Response [200]>
made soup
total_rushing_tds 8 <Response [200]>
made soup
total_rushing_tds 9 <Response [200]>
made soup
total_rush

In [ ]:
player_rows = []
for prop in scraped_json:
    # loop through scraped player pages - 5 players per page
    for i in scraped_json[prop]:

        for j in i['offers']:

            temp = j

            # all data for sinlge player
            # =========================
            individual_player_data = temp

            # meta data for player
            # =========================
            player_meta = individual_player_data['participants'][0]

            pid = player_meta['id']
            name = player_meta['name']
            pos = player_meta['player']['position']
            team = player_meta['player']['team']

            # line and odds data
            # =========================
            opening_line_data = individual_player_data['selections'][0]['opening_line']
            
            opening_line = opening_line_data['line']
            opening_odds = opening_line_data['cost']
            opening_bookid = opening_line_data['book_id']

            # book data 
            # =========================
            maps_bettingpros_books = {
                0:'consensus',
                13:'ceasars',
                10:'fanduel',
                37:'prizepicks',
                19:'betmgm',    
                33:'espnbet',
                27:'party casino',
                49:'hard rock'
            }
            for k in temp['selections'][0]['books']:

                book_id = k['id']
                if book_id != 0:
                    continue
                else:
                    line_data = k['lines'][0]
                    
                    current_odds = line_data['cost']
                    current_line = line_data['line']
                    isMain = line_data['main']
                    isBest = line_data['best']

            #####################
            player_rows.append([
                prop, today, pid, name, pos, team, 
                opening_line, opening_odds, opening_bookid,
                current_odds, current_line
            ])
    


headers = [
    'prop', 'date',
    'playerId', 'name', 'pos', 'team', 
    'opening_line', 'opening_odds', 'opening_bookid', 
    'current_odds', 'current_line'
]
df = pd.DataFrame(
    player_rows,
    columns=headers
)


In [117]:
df

,prop,date,playerId,name,pos,team,opening_ling,opening_odds,opening_bookid,current_odds,current_line
0,total_rushing_yds,2025-08-24,27165,Kaleb Johnson,RB,PIT,900.5,-115,13,-115,713.0
1,total_rushing_yds,2025-08-24,15514,Derrick Henry,RB,BAL,1375.5,-115,13,-110,1325.5
2,total_rushing_yds,2025-08-24,23136,De'Von Achane,RB,MIA,925.5,-114,10,105,875.5
3,total_rushing_yds,2025-08-24,18600,Kyler Murray,QB,ARI,499.5,-115,13,-114,475.5
4,total_rushing_yds,2025-08-24,19792,Chuba Hubbard,RB,CAR,975.5,-115,13,-114,950.5
...,...,...,...,...,...,...,...,...,...,...,...
369,total_passing_tds,2025-08-24,23071,C.J. Stroud,QB,HOU,20.5,-115,33,-118,21.5
370,total_passing_tds,2025-08-24,23084,Caleb Williams,QB,CHI,22.5,-110,19,-140,22.5
371,total_passing_tds,2025-08-24,15600,Dak Prescott,QB,DAL,23.5,-110,19,-105,26.5
372,total_passing_tds,2025-08-24,23046,Drake Maye,QB,NE,19.5,-110,19,-125,20.5


In [96]:
for i in temp['offers'][0]['selections'][0]['books']:
    a = i['lines'][0]
    print(i['id'], a['cost'], a['line'],'\nmain:', a['main'], 'best:',a['best'])

0 -115 3750.5 
main: True best: False
10 -114 3750.5 
main: True best: False
37 -137 3699.5 
main: True best: False
19 -115 3650.5 
main: True best: False
33 -125 3500.5 
main: True best: True
13 -115 3750.5 
main: True best: False
27 -115 3650.5 
main: True best: False
49 -115 3700.5 
main: True best: False
